## Security Context
In Spring Security, `SecurityContext` is a central interface that holds the details of the currently authenticated user for the duration of an execution thread or request. Its primary job is to store `Authentication` object which stores:
<div style="display:inline-block">

| Component           | Description                                                                                                                                             |
|---------------------|---------------------------------------------------------------------------------------------------------------------------------------------------------|
| `Principal`         | Identifies who the user is. Usually an instance of `UserDetails` (containing username, password, enabled status) or a custom user object / JWT payload. |
| `Authorities`       | A collection of `GrantedAuthority` objects representing the user's permissions or roles (e.g., ROLE_ADMIN, READ_PRIVILEGE).                             |
| `Credentials`       | The user's secret (like a password or token). Often erased by Spring Security once authentication is complete to prevent leaking sensitive data.        |
| `Details`           | Web-specific metadata attached to the request, such as the client's IP address or HTTP session ID.                                                      |
| `isAuthenticated()` | A boolean flag indicating whether the current principal has been successfully authenticated (true) or is still unauthenticated (false).                 |    
</div>

`SecurityContextHolder` is the static helper class in Spring Security that acts as the entry point to access, store, and clear the `SecurityContext`. It is the "vault" where Spring Security keeps track of who is currently using the application. Below is an example code where we extract data from `SecurityContextHolder`:

In [ ]:
@Service
public class UserService {

    public String getCurrentUsername() {
        // 1. Get the current SecurityContext
        // 2. Extract the Authentication object
        Authentication authentication = SecurityContextHolder.getContext().getAuthentication();

        if (authentication != null && authentication.isAuthenticated()) {
            return authentication.getName(); // Returns the username
        }
        
        return "Anonymous";
    }
}

The default way `SecurityContextHolder` holds `SecurityContext` is through `ThreadLocal`. This is done in `SecurityContextHolderFilter` whose `doFilter` method looks like:

In [ ]:
private void doFilter(HttpServletRequest request, HttpServletResponse response, FilterChain chain) throws ServletException, IOException {
    Supplier<SecurityContext> deferredContext = this.securityContextRepository.loadDeferredContext(request);
    try {
        this.securityContextHolderStrategy.setDeferredContext(deferredContext);
        chain.doFilter(request, response);
    }
    finally {
        this.securityContextHolderStrategy.clearContext();
    }
}

**SecurityContextHolderStrategy:** interface dictating how and where Spring Security stores the `SecurityContext` in memory during thread execution. Concrete implementations:
1. `ThreadLocalSecurityContextHolderStrategy`: stores the context inside a `ThreadLocal<SecurityContext>`, thus every thread gets its own completely isolated copy of the `SecurityContext`.

In [ ]:
// In this case SecurityContextHolder.getContext() translates to ThreadLocalSecurityContextHolderStrategy.getContext()
// this roughly looks like the code below:

ThreadLocal<SecurityContext> contextHolder = new ThreadLocal<>();

@Override
public SecurityContext> getContext() {
    SecurityContext context = contextHolder.get();
    if (result == null) {
        context = createEmptyContext();
        contextHolder.set(context);
    }

    return context;
}

2. `InheritableThreadLocalSecurityContextHolderStrategy`: similar to previous, the difference being use of `InheritableThreadContext` where child thread spawned from a parent thread inherit the context present in the parent thread.

3. `GlobalSecurityContextHolderStrategy`: one store for entire JVM. Used for standalone desktop GUI applications.

**SecurityContextRepository:** while `SecurityContextHolderStrategy` manages in-memory thread storage during a request, `SecurityContextRepository` handles persistence across multiple HTTP requests. The core interface looks like:

In [ ]:
public interface SecurityContextRepository {
    // Loads context deferred (lazily) for the incoming request
    Supplier<SecurityContext> loadDeferredContext(HttpServletRequest request);
    
    // Explicitly saves the context after authentication
    void saveContext(SecurityContext context, HttpServletRequest request, HttpServletResponse response);
    
    // Checks if a context exists in persistence
    boolean containsContext(HttpServletRequest request);
}

Some implementations:
1. `HttpSessionSecurityContextRepository`: stores the `SecurityContext` inside the servlet container's `HttpSession` under the key `SPRING_SECURITY_CONTEXT`. Used for traditional stateful web applications (e.g., Thymeleaf, JSP, MVC form login).

2. `RequestAttributeSecurityContextRepository`: stores the `SecurityContext` as an HTTP request attribute (`HttpServletRequest.setAttribute()`). This keeps the context available within the scope of a single request execution (including forwards/includes) without persisting it to a session.

3. `DelegatingSecurityContextRepository`: default in Spring Security 6, it combines multiple repositories in order. By default, it delegates to `RequestAttributeSecurityContextRepository` first, then falls back to `HttpSessionSecurityContextRepository`.

4. `NullSecurityContextRepository`: a no-op implementation. It never saves or loads any context. Used for stateless REST APIs (e.g., JWT / OAuth2 Resource Servers) to prevent the container from creating HTTP sessions.

## Authentication Manager
Is a class that attempts to authenticate the passed `Authentication` object, returning a fully populated `Authentication` object (including granted authorities) if successful. It is defined as:

In [ ]:
@FunctionalInterface
public interface AuthenticationManager {

	// The Authentication object returned here is set into the SecurityContext
    Authentication authenticate(Authentication authentication) throws AuthenticationException;
}

`ProviderManager` is a concrete implementation that delegates authentication responsibility to list of `AuthenticationProvider`s. Each one attempts to authenticate the user resulting in success, failure or passing it down the list to other provider (if current one doesn't support).

## Username Password Based Authentication
We start with security config:

In [ ]:
@Bean
public SecurityFilterChain securityFilterChain(HttpSecurity http) throws Exception {
    http
        .authorizeHttpRequests(authorize -> authorize
            .requestMatchers("/login", "/css/**", "/error").permitAll()
            .anyRequest().authenticated()
        )
        // .formLogin(Customizer.withDefaults()) creates a login page for us
        // via DefaultLoginPageGeneratingFilter
        .formLogin(form -> form
            .loginPage("/login")
            .defaultSuccessUrl("/", true)
            .permitAll()
        )
        .logout(logout -> logout
            .logoutUrl("/logout")
            .logoutSuccessUrl("/login?logout")
            .invalidateHttpSession(true)
            .deleteCookies("JSESSIONID")
            .permitAll()
        )
        .sessionManagement(session -> session
            .maximumSessions(1)
        );

    return http.build();
}

The above configuration results in the following filters being placed:
```
DisableEncodeUrlFilter
    ↓
WebAsyncManagerIntegrationFilter
    ↓
SecurityContextHolderFilter
    ↓
HeaderWriterFilter 
    ↓
CsrfFilter 
    ↓
LogoutFilter
    ↓
UsernamePasswordAuthenticationFilter
    ↓
ConcurrentSessionFilter
    ↓
RequestCacheAwareFilter
    ↓
SecurityContextHolderAwareRequestFilter
    ↓
AnonymousAuthenticationFilter
    ↓
SessionManagementFilter
    ↓
ExceptionTranslationFilter
    ↓
AuthorizationFilter
```

### Navigation Paths
**GET /login:** Ignoring the two initial filters, the request would come to `SecurityContextHolderFilter` which would try to look for previously stored `SecurityContext` in `HttpSession` via `SecurityContextRepository` and add it to the `SecurityContextHolder`. If we have not previously authenticated, there would be no `SecurityContext` available.

Ignoring `LogoutFilter` for now (since this is not `/logout`), the request reaches `UsernamePasswordAuthenticationFilter` which roughly has the logic below:

In [ ]:
// UsernamePasswordAuthenticationFilter extends AbstractAuthenticationProcessingFilter
public void doFilter(HttpServletRequest request, HttpServletResponse response, FilterChain chain) throws IOException, ServletException {
    // 1. Only process requests matching the configured login URL.
    if (!request.getRequestURI().equals("/login")
        || !"POST".equalsIgnoreCase(request.getMethod())) {
        // Not a login request → let the next filter handle it. 
        chain.doFilter(request, response);
        return;
    }
    
    try {
        // 2. Extract credentials from the request. POST /login with body username=salman&password=secret 
        String username = request.getParameter("username");
        String password = request.getParameter("password");
        
        if (username == null) { username = ""; }
        if (password == null) { password = ""; }
        
        // 3. Create an unauthenticated Authentication object.
        //    At this point:  authenticated = false, principal = username, credentials = password
        UsernamePasswordAuthenticationToken authenticationRequest = UsernamePasswordAuthenticationToken.unauthenticated(username, password );
        
        // 4. Ask AuthenticationManager to authenticate it.
        //   AuthenticationManager will typically delegate to a AuthenticationProvider -> UserDetailsService -> PasswordEncoder
        Authentication authentication = authenticationManager.authenticate(authenticationRequest);
        
        // 5. Authentication succeeded.
        //    The returned Authentication is now authenticated.
        //    Now, principal = UserDetails("salman"), authorities = [ROLE_USER], authenticated = true
        
        // 6. Create/set the SecurityContext for this request.
        SecurityContext context = SecurityContextHolder.createEmptyContext();
        context.setAuthentication(authentication);
        SecurityContextHolder.setContext(context);
        
        // 7. Persist the SecurityContext.
        //    With normal session-based authentication this eventually results in the context being stored using something like:
        //    HttpSessionSecurityContextRepository which stores it under key SPRING_SECURITY_CONTEXT inside the HttpSession.
        securityContextRepository.saveContext(context, request, response );
        
        // 8. Authentication succeeded. Redirect to defaultSuccessUrl("/home", true)
        successHandler.onAuthenticationSuccess( request, response, authentication );
    } catch (AuthenticationException ex) {
        // 9. Authentication failed. For example: a) user doesn't exist b) password doesn't match c) account is disabled
        //    The configured failureUrl("/login?error") causes the configured failure handler to redirect the user
        SecurityContextHolder.clearContext();
        failureHandler.onAuthenticationFailure(request, response, ex );
    }
}

Given this is GET request, the above filter would be skipped.

Request passes through bunch of other filters and lands on `AuthorizationFilter`. This filter is also skipped since our security configuration says `/login` doesn't need authorization.

**POST /login:** `SecurityContextHolderFilter` behaves same as earlier, the interesting bits are in `UsernamePasswordAuthenticationFilter`. The line `authenticationManager.authenticate(authenticationRequest)` requires an instance of `AuthenticationManager`.

If we provide a `UserDetailsService` bean and a `PasswordEncoder` bean Spring Boot automatically configures a `AuthenticationProvider` (more specifically `DaoAuthenticationProvider`) bean for us. This provider is used in the authentication manager constructed also by Spring Boot.

In [ ]:
@Bean
public UserDetailsService userDetailsService(DataSource dataSource) {
    return new JdbcUserDetailsManager(dataSource);
}

@Bean
public PasswordEncoder passwordEncoder() {
    return new BCryptPasswordEncoder();
}

If the authentication is successful, the security filter chain is not continued forward and request is redirected to `.defaultSuccessUrl("/", true)`. On failure as well, request is short circuited and forwarded to `failureUrl` (not mentioned in the security config above, thus the default is `/login?error`).

**GET /order:** If the user had not logged in earlier, then:
1. Request lands at `SecurityContextHolderFilter`, it looks for a `SecurityContext` in the `HttpSession` via `SecurityContextRepository`. Finds nothing.
2. Passes through `CsrfFilter`, `LogoutFilter`, etc to land at `UsernamePasswordAuthenticationFilter` where it is a no-op since it is not `/login` path
3. Passes through some other filters and reaches `AuthorizationFilter`. Here since `SecurityContext` contains no valid authentication, it results in `AccessDeniedException`
4. Request now reaches `ExceptionTranslationFilter` where it invokes the `AuthenticationEntryPoint` and gets redirected to `/login`

If the request contained `JSESSIONID`:
1. Request lands at `SecurityContextHolderFilter`, it looks for a `SecurityContext` in the `HttpSession` via `SecurityContextRepository`. It finds the previously-stored `SecurityContext` (the authenticated `UsernamePasswordAuthenticationToken` (which is implementation of `Authentication`)). It then loads the `SecurityContext` into `SecurityContextHolder`
2. Passes through `CsrfFilter`, `LogoutFilter`, etc to land at `UsernamePasswordAuthenticationFilter` where it is a no-op since it is not `/login` path
3. Passes through some other filters and reaches `AuthorizationFilter`. Here it checks `SecurityContextHolder.getContext().getAuthentication()` it finds the real, authenticated token (`isAuthenticated() == true`, real authorities)
4. Request is allowed to reach the intended `@Controller`.

### Logging Out
`LogoutFilter` handles logout requests, it performs the following two operations:
- `request.getSession().invalidate()`
- `SecurityContextHolder.clearContext()`
- Clears `JSESSINID` cookie by setting cookie age as 0.

In [ ]:
@Override
public void doFilter(ServletRequest req, ServletResponse res, FilterChain chain)
        throws IOException, ServletException {

    HttpServletRequest request = (HttpServletRequest) req;
    HttpServletResponse response = (HttpServletResponse) res;

    if (!logoutRequestMatcher.matches(request)) {
        chain.doFilter(request, response); // not a logout request, pass through
        return;
    }

    Authentication auth = SecurityContextHolder.getContext().getAuthentication();
    this.handler.logout(request, response, auth); // delegates to composite handler chain
    this.logoutSuccessHandler.onLogoutSuccess(request, response, auth); // redirect/response
    // note: no chain.doFilter() call here — the chain terminates, response is committed
}

`CompositeLogoutHandler` uses multiple `LogoutHandler`s to perform logout:

In [ ]:
// SecurityContextLogoutHandler — the core one
public class SecurityContextLogoutHandler implements LogoutHandler {
    private boolean invalidateHttpSession = true;

    public void logout(HttpServletRequest request, HttpServletResponse response, Authentication auth) {
        if (invalidateHttpSession) {
            HttpSession session = request.getSession(false); // false = don't create one if absent
            if (session != null) {
                session.invalidate(); // destroys the session server-side entirely
            }
        }
        SecurityContextHolder.clearContext(); // clears the thread-local
    }
}

// CookieClearingLogoutHandler — clears JSESSIONID (and any other configured cookies)
public class CookieClearingLogoutHandler implements LogoutHandler {
    private final List<String> cookiesToClear; // e.g. "JSESSIONID"

    public void logout(HttpServletRequest request, HttpServletResponse response, Authentication auth) {
        for (String cookieName : cookiesToClear) {
            Cookie cookie = new Cookie(cookieName, null);
            cookie.setMaxAge(0);       // expire immediately
            cookie.setPath(getCookiePath(request));
            response.addCookie(cookie);
        }
    }
}

### Persistent Session
By default sessions are stored in memory, this means that whenever the application is restarted all session information is lost. To persist session we can use Spring Session using `spring-boot-started-session-jdbc` repository. Spring Boot automatically creates `SPRING_SESSION` and `SPRING_SESSION_ATTRIBUTES` tables (which store `SecurityContext` as bytes):
<div style="display:inline-block">

| primary_id	                        | session_id	                      | creation_time | last_access_time | max_inactive_interval | expiry_time	 | principal_name |
|---------------------------------------|-------------------------------------|---------------|------------------|-----------------------|---------------|----------------|
| cd2ab69b-7f0e-4793-8d6f-e742277dfcbf  | 28c91cb8-9d1c-4290-b57c-dccb8a24aa04|	1786464884687 | 1786464891732	 | 1800	                 | 1786466691732 | admin          |
</div>
<div style="display:inline-block">

|session_primary_id	                  | attribute_name	                                                                | attribute_bytes |
|-------------------------------------|---------------------------------------------------------------------------------|-----------------|
|cd2ab69b-7f0e-4793-8d6f-e742277dfcbf |	SPRING_SECURITY_CONTEXT	                                                        | binary data     |
|cd2ab69b-7f0e-4793-8d6f-e742277dfcbf |	org.springframework.security.web.csrf.HttpSessionCsrfTokenRepository.CSRF_TOKEN	| binary data     |
</div>

The `SESSION` (not `JSESSIONID` - Spring Session replaces that) cookie created by Spring Session gets removed when the browser is closed, this means that user is effectively logged out on browser getting closed. So ideally we should set the cookie with long expiry time and control session lifetime through the database entry in the `SPRING_SESSION`:
```yml
# Cookie persists across browser restarts (1 year in seconds)
server.servlet.session.cookie.max-age=31536000
# Session expiry controlled server-side (30 min of inactivity)
server.servlet.session.timeout=30m
```

### Remember Me
To understand remember me, let's first consider the case when we didn't have Spring Session. The default setting for `JSESSIONID` is that it lasts as long as the browser window is open. Thus closing browser window terminated the session. Remember me enabled us to have long living session persisting across multiple browser session. It does that by introducing another cookie which is long lived. Spring can authenticate the user using this long lived cookie even when `JSESSIONID` is missing in the request.

Following code introduces remember me functionality:

In [ ]:
// Invoked on HttpSecurity object
.rememberMe(remember -> remember
    // Repository where the long lived token is stored
    .tokenRepository(persistentTokenRepository(http.getSharedObject(DataSource.class)))
    .tokenValiditySeconds(604800) // 7 days
)

When we login with remember me option turned on, `UsernamePasswordAuthenticationFilter` uses `RememberMeService`. It is an interface/strategy used by Spring Security to implement the remember me lifecycle. It is called during successful login, later when Spring Security tries to auto-login from the remember-me cookie, and during logout. Two well known implementations:
- `TokenBasedRememberMeServices`: stores a signed token in the cookie. No remember-me database required.
- `PersistentTokenBasedRememberMeServices`: stores a series/token in a database and puts the corresponding token in the cookie. Tokens are rotated on use.

In `UsernamePasswordAuthenticationFilter` after successful login, `RememberMeService.loginSuccess` is called. Depending upon implementation of `RememberMeService` it:
- `TokenBasedRememberMeServices`: sets a long living cookie named `remember-me` containing username and expiry time signed using a secret (user's password and a constant key).
- `PersistentTokenBasedRememberMeServices`: it uses the below table:
  ```sql
  CREATE TABLE persistent_logins (
    username VARCHAR(64) NOT NULL,
    series VARCHAR(64) PRIMARY KEY, -- identifies "this device/browser instance," stays constant. Set to some random bytes
    token VARCHAR(64) NOT NULL, -- rotates on every use. Set to some random bytes
    last_used TIMESTAMP NOT NULL
  );
  ```
  On successful login, a new entry is made into the above table and also a cookie named `remember-me` is created with value set to the series and token data above.

Now, lets say a request is made to any endpoint without the `JSESSIONID` but with `remember-me` cookie. Spring Security has added `RememberMeAuthenticationFilter` just above the authorization filter. It roughly looks like:

In [ ]:
@Override
public void doFilter(HttpServletRequest req, HttpServletResponse res, FilterChain chain) throws IOException, ServletException {

    // 1. Only act if nothing has already authenticated this request, meaning JSESSIONID (or SESSION) cookie was missing
    if (SecurityContextHolder.getContext().getAuthentication() == null) {

        // 2. Delegate to whichever RememberMeServices is configured
        Authentication rememberMeAuth = this.rememberMeServices.autoLogin(request, response);

        if (rememberMeAuth != null) {
            try {
                // 3. Attach request details (IP, session ID) before validating
                rememberMeAuth.setDetails(this.authenticationDetailsSource.buildDetails(request));

                // 4. Run it through the AuthenticationManager — NOT DaoAuthenticationProvider, but RememberMeAuthenticationProvider, 
                //    which just checks the token's embedded key matches the configured key and the account isn't disabled/locked 
                //    — no password re-verification happens here
                Authentication authResult = this.authenticationManager.authenticate(rememberMeAuth);

                // 5. Populate the context, same effect as any other successful auth filter
                SecurityContext context = SecurityContextHolder.createEmptyContext();
                context.setAuthentication(authResult);
                SecurityContextHolder.setContext(context);

                // 6. Add to session cookie
                this.securityContextRepository.saveContext(context, request, response);

                onSuccessfulAuthentication(request, response, authResult);

                if (this.eventPublisher != null) {
                    eventPublisher.publishEvent(
                        new InteractiveAuthenticationSuccessEvent(authResult, this.getClass()));
                }

            } catch (AuthenticationException failed) {
                // e.g. account disabled/locked, or key mismatch
                SecurityContextHolder.clearContext();
                onUnsuccessfulAuthentication(request, response, failed);
            }
        }
    }

    chain.doFilter(request, response);
}

Again, `autoLogin` depends upon the `RememberMeServices` implementation:
- `TokenBasedRememberMeServices`: works roughly as:

In [ ]:
public Authentication autoLogin(HttpServletRequest request, HttpServletResponse response) {
    String cookieValue = extractRememberMeCookie(request);
    String[] cookieTokens = decodeCookie(cookieValue); // [username, expiryTime, signature]

    String username = cookieTokens[0];
    long tokenExpiryTime = Long.parseLong(cookieTokens[1]);

    if (isTokenExpired(tokenExpiryTime)) {
        cancelCookie(request, response);
        return null;
    }

    UserDetails userDetails = getUserDetailsService().loadUserByUsername(username);

    // recompute what the signature SHOULD be, given the current username/expiry/password/key
    String expectedSignature = makeTokenSignature(tokenExpiryTime, username, userDetails.getPassword());

    if (!equals(expectedSignature, cookieTokens[2])) { // constant-time comparison
        throw new InvalidCookieException("Cookie token[2] contained signature '" + cookieTokens[2] +
                "' but expected '" + expectedSignature + "'");
    }

    return createSuccessfulAuthentication(request, userDetails);
}

- `PersistentTokenBasedRememberMeServices`: its `autoLogin` looks like:

In [ ]:
public Authentication autoLogin(HttpServletRequest request, HttpServletResponse response) {
    String cookieValue = extractRememberMeCookie(request);
    String[] cookieTokens = decodeCookie(cookieValue); // [series, token]

    // Look up the series
    PersistentRememberMeToken stored = tokenRepository.getTokenForSeries(cookieTokens[0]);
    if (stored == null) {
        throw new RememberMeAuthenticationException("No token found for series: " + series);
    }

    // Theft check - presented token must match the current stored value. Why? Let attacker steal remember me cookie (Series=S1, Token=T1)
    // It uses above to login and is issued a new remember me cookie (Series=S1, Token=T2). Now the legitimate user uses the cookie
    // (Series=S1, Token=T1) thus trigerring theft detection
    if (!cookieValue[1].equals(stored.getTokenValue())) {
        // wipe all sessions in persistent_logins, treat as compromised. Remember that the normal JSESSIONID (or SESSION) based sessions are
        // not cleared here. So attacker would still have access to application. Potential ISSUE??
        tokenRepository.removeUserTokens(stored.getUsername());
        throw new CookieTheftException("Token mismatch — possible cookie theft");
    }

    if (isExpired(stored)) {
        throw new RememberMeAuthenticationException("Remember-me token expired");
    }

    // Rotate the token (old value now becomes a tripwire) and update the cookie
    String newTokenValue = generateTokenValue();
    tokenRepository.updateToken(series, newTokenValue, new Date());
    setCookie(new String[]{ series, newTokenValue }, tokenValiditySeconds, request, response);

    // Build the Authentication
    UserDetails user = userDetailsService.loadUserByUsername(stored.getUsername());
    return new RememberMeAuthenticationToken(key, user, user.getAuthorities());
}

On logout, via `LogoutFilter`,
- `TokenBasedRememberMeServices`: just clears the cookie (Max-Age=0) client-side
- `PersistentTokenBasedRememberMeServices`: logout does two things a) clears the cookie client-side b) deletes the row(s) from `persistent_logins` server-side (via `tokenRepository.removeUserTokens(username)` or similar).

Is there a simpler way to add remember-me when we are using Spring Session JDBC based session persistence? One way could be to simply change the expiry time in the `SPRING_SESSION` table based on whether the user had selected remember me or not. This is what another implementation of `RememberMeServices`, which is `SpringSessionRememberMeServices` does.

### Limiting Concurrent Sessions
To limit number of concurrent logins, we can:

In [ ]:
http.sessionManagement(session -> session
    .maximumSessions(1)                    // how many concurrent sessions per user
    .maxSessionsPreventsLogin(false)       // false = kick out oldest; true = reject new login
    .expiredUrl("/login?expired")
);

To control number of concurret sessions, we need some time of registry to store mapping of sessions to a user. Spring Security provides:

In [ ]:
public interface SessionRegistry {
	List<Object> getAllPrincipals();
	List<SessionInformation> getAllSessions(Object principal, boolean includeExpiredSessions);
	@Nullable SessionInformation getSessionInformation(String sessionId);    
	void registerNewSession(String sessionId, Object principal);
	void removeSessionInformation(String sessionId);
}

An implementation is `SessionRegistryImpl` which is the default implementation. When Spring Session is involved, we would use `SpringSessionBackedSessionRegistry`. `SessionRegistryImpl` works by maintaining two maps:

In [ ]:
// <principal:Object,SessionIdSet>
private final ConcurrentMap<Object, Set<String>> principals;
// <sessionId:Object,SessionInformation>
private final Map<String, SessionInformation> sessionIds;

`SpringSessionBackedSessionRegistry` on the other hand uses session repository (session tables in case of Spring Session JDBC).

`UsernamePasswordAuthenticationFilter` on new login checks if there are existing logins for the user and performs further action. This is controlled via `ConcurrentSessionControlAuthenticationStrategy` an implementation of `SessionAuthenticationStrategy`. `SessionAuthenticationStrategy` runs inside `UsernamePasswordAuthenticationFilter` after successful authentication. It uses session repository to check how many concurrent sessions the user has. If the count < max, then do nothing. If it is equal to max, then check current session against all stored sessions.

**Expired Sessions:** `ConcurrentSessionFilter` is another component in the whole flow, it runs before `UsernamePasswordAuthenticationFilter` on every request and does:

In [ ]:
private void doFilter(HttpServletRequest request, HttpServletResponse response, FilterChain chain)
			throws IOException, ServletException {
    HttpSession session = request.getSession(false);
    if (session != null) {
        SessionInformation info = this.sessionRegistry.getSessionInformation(session.getId());
        if (info != null) {
            if (info.isExpired()) {
                // Expired - abort processing
                this.logger.debug(LogMessage
                    .of(() -> "Requested session ID " + request.getRequestedSessionId() + " has expired."));
                doLogout(request, response);
                this.sessionInformationExpiredStrategy
                    .onExpiredSessionDetected(new SessionInformationExpiredEvent(info, request, response, chain));
                return;
            }
            // Non-expired - update last request date/time
            this.sessionRegistry.refreshLastRequest(info.getSessionId());
        }
    }
    chain.doFilter(request, response);
}

## Kerberos Authentication
### Protocol Overview
To understand how Kerberos authentication works, we first need to discuss the three components that are involved:
- client: user that is trying to access
- KDC (Key Distribution Center): trusted authority
- resource server: the service the client is trying to access

![Kerbero Auth Steps](./images/kerberos_authentication.png)

The three steps are summarised below:
1. Step 1: client makes `AS-REQ` which looks like:
   <pre style="background-color: #f4f4f5; color: #18181b; padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
    AS-REQ {
      pvno: 5                          // Kerberos protocol version
      msg-type: 10                     // AS-REQ
      padata: [                        // pre-authentication data
        { 
          type: PA-ENC-TIMESTAMP,
          value: Enc(timestamp, client_password_key) 
        }
      ]
      req-body: {
        cname: "alice"                 // client principal (username)
        realm: "EXAMPLE.COM"           // Kerberos realm/domain
        sname: "krbtgt/EXAMPLE.COM"    // service requested = the TGS itself
        till: "20260817120000Z"        // requested ticket expiry
        nonce: 837462910                // random number, matched in reply to prevent replay
        etype: [AES256-CTS-HMAC-SHA1]  // supported encryption types
      }
    }
   </pre>   

   KDC server decrypts the `PA-DATA` (again using `client_password_key`, as KDC as stored user's password). If KDC is able to decrypt then it means it has established client's identity. The response `AS-REP` looks like:
   <pre style="background-color: #f4f4f5; color: #18181b; padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
   AS-REP {
      pvno: 5
      msg-type: 11
      cname: "alice"
      crealm: "EXAMPLE.COM"
      ticket: {                         // the TGT can't be decrypted by the client
        tkt-vno: 5
        realm: "EXAMPLE.COM"
        sname: "krbtgt/EXAMPLE.COM"
        enc-part: Enc({                 // encrypted with KDC's own secret key
          flags: [forwardable, renewable]
          key: SESSION_KEY_1            // session key, copy #2
          crealm: "EXAMPLE.COM"
          cname: "alice"
          authtime: "...", starttime: "...", endtime: "..."
          caddr: [client IP]            // optional address restriction
        }, krbtgt_secret_key)
      }
      enc-part: Enc({                   // encrypted with client's password-derived key
        key: SESSION_KEY_1              // session key, copy #1, client can read this one
        nonce: 837462910                // echoed back, must match request
        last-req: [...]
        authtime, endtime, renew-till
        srealm, sname: "krbtgt/EXAMPLE.COM"
      }, client_password_key)
    }
   </pre>
   This step essesntially means that the KDC has authenticated Alice, and the KDC is willing to issue tickets on Alice's behalf.

3. Step 2: client sends a `TGS-REQ` to KDC which looks like:
   <pre style="background-color: #f4f4f5; color: #18181b; padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
   TGS-REQ {
      pvno: 5
      msg-type: 12
      padata: [
        { type: PA-TGS-REQ,
          value: {
            ap-req: {                          // the client "presents" the TGT here
              ticket: <TGT from AS-REP>,
              authenticator: Enc({
                cname: "alice"
                crealm: "EXAMPLE.COM"
                ctime: "20260816120500Z"       // current timestamp, proves freshness
              }, SESSION_KEY_1)
            }
          }
        }
      ]
      req-body: {
        cname: "alice"
        sname: "cifs/fileserver01.example.com" // the actual service being requested
        till: "20260816220000Z"
        nonce: 129384756
      }
    }
   </pre>


   Ticket generating server compares user information present in TGT ticket and authenticator. After this it generates another session keys. It returns response like:
   <pre style="background-color: #f4f4f5; color: #18181b; padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
   TGS-REP {
      pvno: 5
      msg-type: 13
      cname: "alice"
      ticket: {                          // the Service Ticket, opaque to client
        sname: "cifs/fileserver01.example.com"
        enc-part: Enc({                  // encrypted with the FILE SERVER's secret key
          flags: [forwardable]
          key: SESSION_KEY_2             // new session key, copy #2
          cname: "alice", crealm: "EXAMPLE.COM"
          authtime, starttime, endtime
        }, fileserver_secret_key)
      }
      enc-part: Enc({                    // encrypted with SESSION_KEY_1 (from step 2)
        key: SESSION_KEY_2               // new session key, copy #1 — client reads this
        nonce: 129384756
        sname: "cifs/fileserver01.example.com"
        endtime
      }, SESSION_KEY_1)
    }
   </pre>

5. Lastly, the client uses the service ticket and authenticator (created using client-service session key) and sends it to the service for authentication purpose:
   <pre style="background-color: #f4f4f5; color: #18181b; padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
    AP-REQ {
      pvno: 5
      msg-type: 14
      ap-options: [mutual-required]      // client is asking for mutual authentication
      ticket: <Service Ticket from TGS-REP>
      authenticator: Enc({
        cname: "alice"
        crealm: "EXAMPLE.COM"
        ctime: "20260816120510Z"         // fresh timestamp, proves this isn't a replay
        subkey: OPTIONAL_SESSION_SUBKEY  // optional, for the app-level session
      }, SESSION_KEY_2)
    }
   </pre>

   Responds with:
   <pre style="background-color: #f4f4f5; color: #18181b; padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
    AP-REP {
      pvno: 5
      msg-type: 15
      enc-part: Enc({
        ctime: "20260816120510Z"         // echoes the client's own timestamp back
        cusec: ...
        subkey: OPTIONAL_SESSION_SUBKEY  // may confirm/replace the session subkey
        seq-number: ...                  // for ordering subsequent messages
      }, SESSION_KEY_2)
    }
   </pre>
   `AP-REP` can be used by client to confirm that the service is what it claims to be.

### Keytab File
When a human is the client, then he can type in the password on `kinit`. What if it is an automated service acting as a client? It contains principal (corresponding to the user) and a secret key (generated using the user's password). This keytab file can be used in lieu of password like:
```bash
$ kinit -kt alice.keytab alice@EXAMPLE.COM
```

Keytabs are also used by service, but in a different context. It is used to decrypt the service ticket sent to it as this keytab holds the same secret key that the KDC/TGS used, back when it encrypted a Service Ticket. Normally, the service and KDC do not interact with each other, though periodically, if the service account's password changes, a new keytab needs to be generated and deployed which does involve KDC interaction.

The service itself may interact with another Kerberos protected service, in which case the keytab file is used in AS exchange.

### krb5.conf
When we do `kinit` how does the command know the location of the KDC infrastructure? The answer is `krb5.conf` file. A sample looks like:
<pre style="margin: 1em 0; background-color: #f4f4f5; color: #18181b; padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
[libdefaults]
    default_realm = EXAMPLE.COM

[realms]
    EXAMPLE.COM = {
        kdc = dc01.example.com
    }

[domain_realm]
    .example.com = EXAMPLE.COM
</pre>
This suggests that for realm `EXAMPLE.COM`, the KDC server is located on `dc01.example.com`. 

In a Java application, we can provide the location of this file using VM arg: `-Djava.security.krb5.conf=/path/to/krb5.conf`. In Linux it is present by default in `/etc/krb5.conf`.

A pure Kerberos service doesn't really need it since the service doesn't need to talk to KDC to authenticate a request.

### HTTP Authentication
**SPNEGO (Simple and Protected GSS-API Negotiation Mechanism):** is a negotiation wrapper that lets a client and server agree on which authentication mechanism to use, commonly Kerberos. Server lists the authentication mechanism it supports, client compares it with authentication mechanisms it supports and both negotiate.

Now let's see how kerberos authentication is done in context of an HTTP service. The steps are:
1. Client logs in and runs `kinit`. This performs AS exchange:
   <pre style="background-color: #f4f4f5; color: #18181b; padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
    AS-REQ
        cname: alice, realm: EXAMPLE.COM
        sname: krbtgt/EXAMPLE.COM
        padata: Enc(timestamp, client_password_key)
   </pre>
   <pre style="background-color: #f4f4f5; color: #18181b; padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
    AS-REP
        ticket (TGT): Enc({key: SESSION_KEY_1, cname: alice, endtime: ...}, krbtgt_secret_key)
        enc-part: Enc({key: SESSION_KEY_1, nonce, endtime}, client_password_key)
   </pre>
   `TGT` and `SESSION_KEY_1` is cached in the machine in location `/tmp/krb5cc_*`. This cache can be used for next few hours.

2. Browser makes unauthenticated request:
   ```http
    GET /reports/quarterly.html HTTP/1.1
    Host: intranet.example.com
   ```
   ```http
    HTTP/1.1 401 Unauthorized
    WWW-Authenticate: Negotiate
    WWW-Authenticate: Kerberos
    Content-Length: 0
   ```
   Negotiate in the response activates `SPNEGO` mechanism. Client and server agree on Kerberos authentication.

3. Client's Kerberos stack does the TGS Exchange (the browser / OS if request originated in the browser):
   <pre style="background-color: #f4f4f5; color: #18181b; padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
    TGS-REQ
        ticket: <TGT from Step 0>,
        authenticator: Enc({cname: alice, ctime: now}, SESSION_KEY_1)
        ...
   </pre>
   <pre style="background-color: #f4f4f5; color: #18181b; padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
    TGS-REP
        ticket (Service Ticket): Enc({key: SESSION_KEY_2, cname: alice, endtime}, webserver_secret_key)
        enc-part: Enc({key: SESSION_KEY_2, nonce, sname: HTTP/intranet.example.com}, SESSION_KEY_1)
   </pre>
   At this point `Service Ticket` and `SESSION_KEY_2` are cached for this specific service, reusable for future requests to the same site.


4. The `AP-REQ` is wrapped in a SPNEGO NegTokenInit, base64-encoded, and placed in the `Authorization` header.
   ```http
    GET /reports/quarterly.html HTTP/1.1
    Host: intranet.example.com
    Authorization: Negotiate YIIFvQYGKwYBBQUCoIIFsTCCBa2gMDAuBgkqhkiG9xIBAgIC
        AG4jggWfBIIFm2eBmDCCBZSgAwIBBaEDAgEOogcDBQAgAAAAo4IEZWGCBGEwggRdoAMC
        ARWhDBsKRVhBTVBMRS5DT02iIzAhoAMCAQGhGjAYGwRIVFRQGxBpbnRyYW5ldC5leGFt
        cGxlo4IEIzCCBB+gAwIBEqEDAgECooIEEQSCBA0...
    ```
    If we decode the above code, we would get:
    <pre style="padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
    NegTokenInit {
      mechTypes: [Kerberos-v5-OID, NTLMSSP-OID]
      mechToken: AP-REQ {
        ap-options: [mutual-required]
        ticket: <Service Ticket, opaque, from Step 3>
        authenticator: Enc({cname: alice, ctime: now}, SESSION_KEY_2)
      }
    }
    </pre>

5. Server validates the request and returns the response. The response may optionally contain `AP-REP`:
   ```http
    HTTP/1.1 200 OK
    WWW-Authenticate: Negotiate oYHwMIHtoAMKAQChCwYJKoZIhvcSAQICooHZBIHWY4HT
        MIHQoAMCARWhAwIBA6KBwwSBwGVQNfEwe...
    Content-Type: text/html
    Content-Length: 4521
    
    <html>...quarterly report content...</html>
   ```
   Decoding, we get
   <pre style="padding: 12px; border-radius: 6px; border: 1px solid #e4e4e7;">
   NegTokenResp {
      negState: accept-completed
      supportedMech: Kerberos-v5-OID
      responseToken: AP-REP {
        enc-part: Enc({ctime: <echoed from Step 4>}, SESSION_KEY_2)
      }
    }
   </pre>
   
6. Client verifies the `AP-REP` (if mutual authentication was requested originally).

### Spring Security Kerberos
Spring Security provides support for Kerberos. To demonstrate, we can add Kerberos authentication on top of username password based authentication we discussed before.

One component is `SpnegoAuthenticationProcessingFilter` defined loosely as:

In [ ]:
public void doFilter(HttpServletRequest request, HttpServletResponse response, FilterChain chain) {
    // 1. Skip this filter if the user is already authenticated, check
    // SecurityContextHolder.getAuthentication()

    String header = request.getHeader("Authorization");

    // 2. Only proceed if header is a genuine Negotiate/Kerberos token
    if (header != null && header.startsWith("Negotiate ")) {

        // 3. Decode the base64 SPNEGO/AP-REQ token
        byte[] token = Base64.decode(header.substring(header.indexOf(' ') + 1).getBytes());

        try {
            // 4. Wrap raw bytes into Spring Security's Kerberos-specific Authentication object
            //    KerberosServiceRequestToken implements Authentication 
            KerberosServiceRequestToken authenticationRequest = new KerberosServiceRequestToken(token);
            authenticationRequest.setDetails(authenticationDetailsSource.buildDetails(request));

            // 5. Delegate actual ticket validation to the AuthenticationManager,
            //    which internally calls KerberosServiceAuthenticationProvider —
            //    this is where the keytab-based decryption + authenticator
            //    check actually happens (via sun.security.jgss / GSSManager)
            Authentication authentication = authenticationManager.authenticate(authenticationRequest);

            // 6. Update session-related state if configured (e.g. session fixation protection)
            sessionStrategy.onAuthentication(authentication, request, response);

            // 7. Populate SecurityContext — request is now authenticated
            SecurityContext context = securityContextHolderStrategy.createEmptyContext();
            context.setAuthentication(authentication);
            securityContextHolderStrategy.setContext(context);

            // 8. Add authentication data to session
            this.securityContextRepository.saveContext(context, request, response);

            if (successHandler != null) {
                successHandler.onAuthenticationSuccess(request, response, authentication);
            }

        } catch (AuthenticationException e) {
            // 9. Ticket invalid/expired/decrypt failure
            SecurityContextHolder.clearContext();
            if (failureHandler != null) {
                failureHandler.onAuthenticationFailure(request, response, e);
                return;
            } else {
                throw e; // propagates up to ExceptionTranslationFilter -> SpnegoEntryPoint -> re-challenge
            }
        }
    }

    // 10. Continue the chain regardless — if no header was present at all,
    //    this filter does nothing; it's SpnegoEntryPoint (triggered later
    //    on an AccessDeniedException/AuthenticationException) that actually
    //    sends the 401 + WWW-Authenticate: Negotiate challenge
    chain.doFilter(request, response);
}

We update `SecurityConfig` to add some additional beans as:

In [ ]:
@Bean
public DaoAuthenticationProvider daoAuthenticationProvider(
        UserDetailsService userDetailsService,
        PasswordEncoder passwordEncoder) {
    DaoAuthenticationProvider daoProvider = new DaoAuthenticationProvider(userDetailsService);
    daoProvider.setPasswordEncoder(passwordEncoder);
    return daoProvider;
}

@Bean
public KerberosServiceAuthenticationProvider kerberosServiceAuthenticationProvider(UserDetailsService userDetailsService) {
    KerberosServiceAuthenticationProvider provider = new KerberosServiceAuthenticationProvider();
    provider.setTicketValidator(sunJaasKerberosTicketValidator());
    
    // Wrap the JDBC UserDetailsService to strip the domain from the Kerberos principal
    provider.setUserDetailsService(username -> {
        String bareUsername = username.contains("@")
                ? username.substring(0, username.indexOf('@'))
                : username;
        return userDetailsService.loadUserByUsername(bareUsername);
    });
    
    return provider;
}

@Bean
public SpnegoEntryPoint spnegoEntryPoint() {
    return new SpnegoEntryPoint("/login");
}

public SpnegoAuthenticationProcessingFilter spnegoAuthenticationProcessingFilter(
        AuthenticationManager authenticationManager) {
    SpnegoAuthenticationProcessingFilter filter = new SpnegoAuthenticationProcessingFilter();
    filter.setAuthenticationManager(authenticationManager);
    return filter;
}

@Bean
public SunJaasKerberosTicketValidator sunJaasKerberosTicketValidator() {
    SunJaasKerberosTicketValidator ticketValidator = new SunJaasKerberosTicketValidator();
    ticketValidator.setServicePrincipal(servicePrincipal);
    ticketValidator.setKeyTabLocation(new FileSystemResource(keytabLocation));
    ticketValidator.setDebug(true);
    return ticketValidator;
}

What if a service needs to communicate with another service authenticated through Kerberos? We use `KerberosRestTemplate`.